<a href="https://colab.research.google.com/github/simulate111/TOPSIS_Thes/blob/main/TOPSIS%2BTEA%2BLCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

# This will open an upload dialog below the cell
uploaded = files.upload()

Saving LCA.csv to LCA.csv


In [2]:
import pandas as pd
import numpy as np

# ===============================
# STEP 1 & 2: LOAD CSV & EXTRACT CLEAN ROWS
# ===============================
# Read "LCA.csv" verbatim. We use latin1 encoding to handle special characters in city names.
df_raw = pd.read_csv("LCA.csv", encoding="latin1")

# Map the exact CSV column headers to the clean names required for TOPSIS
column_mapping = {
    'WT': 'City',
    'LCOE': 'LCOE_Wind',
    'CO\-(2)-eq./kWh': 'CO2_Wind',
    'LCOE.1': 'LCOE_NoWind',
    'CO\-(2)-eq./kWh.1': 'CO2_NoWind',
    'LCOE.2': 'LCOE_PV',
    '\\f:Times New Roman(CO)\\-(\\f:Times New Roman(2))\\f:Times New Roman(-eq./kWh)': 'CO2_PV'
}

# Apply the mapping, rename columns, and drop the first two rows (metadata/units)
df = df_raw[list(column_mapping.keys())].rename(columns=column_mapping).iloc[2:].copy()

# Drop rows without a valid City name
df = df.dropna(subset=['City'])

# Convert all cost and emission columns to numeric floats, dropping any invalid rows
numeric_cols = ["LCOE_Wind", "CO2_Wind", "LCOE_NoWind", "CO2_NoWind", "LCOE_PV", "CO2_PV"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=numeric_cols).reset_index(drop=True)

print("Cleaned data preview:")
print(df.head())


# ===============================
# STEP 3: TOPSIS FUNCTION
# ===============================
def topsis(X, weights, criteria):
    X = np.array(X, dtype=float)
    weights = np.array(weights)
    criteria = np.array(criteria)

    norm = np.sqrt((X**2).sum(axis=0))
    Y = X / norm
    Yw = Y * weights

    Vp = np.where(criteria == 1, Yw.max(axis=0), Yw.min(axis=0))
    Vn = np.where(criteria == 1, Yw.min(axis=0), Yw.max(axis=0))

    Splus = np.sqrt(((Yw - Vp)**2).sum(axis=1))
    Snegative = np.sqrt(((Yw - Vn)**2).sum(axis=1))

    return Snegative / (Splus + Snegative)


# ===============================
# STEP 4: RUN TOPSIS FOR ALL CITIES
# ===============================
weights = [0.5, 0.5]
criteria = [0, 0]  # both cost

results = []

for _, row in df.iterrows():
    X = [
        [row["LCOE_Wind"], row["CO2_Wind"]],
        [row["LCOE_NoWind"], row["CO2_NoWind"]],
        [row["LCOE_PV"], row["CO2_PV"]],
    ]

    scores = topsis(X, weights, criteria)
    best_idx = np.argmax(scores)

    scenario_names = ["Wind", "NoWind", "PV"]

    results.append({
        "City": row["City"],
        "Best": scenario_names[best_idx],
        "Scores": scores
    })


# ===============================
# STEP 5: OUTPUT
# ===============================
results_df = pd.DataFrame(results)

print("\nFinal Results:")
print(results_df.head(10))


# ===============================
# STEP 6: DOWNLOAD RESULTS
# ===============================
results_df.to_excel("topsis_results.xlsx", index=False)

from google.colab import files
files.download("topsis_results.xlsx")

<>:14: SyntaxWarning: invalid escape sequence '\-'
<>:16: SyntaxWarning: invalid escape sequence '\-'
<>:14: SyntaxWarning: invalid escape sequence '\-'
<>:16: SyntaxWarning: invalid escape sequence '\-'
/tmp/ipykernel_2069/3336139846.py:14: SyntaxWarning: invalid escape sequence '\-'
  'CO\-(2)-eq./kWh': 'CO2_Wind',
/tmp/ipykernel_2069/3336139846.py:16: SyntaxWarning: invalid escape sequence '\-'
  'CO\-(2)-eq./kWh.1': 'CO2_NoWind',


Cleaned data preview:
        City  LCOE_Wind    CO2_Wind  LCOE_NoWind  CO2_NoWind   LCOE_PV  \
0      Turku   0.346154   95.842278     0.533846  171.664504  1.692308   
1   Helsinki   0.260000   69.848794     0.555385  203.718880  1.707692   
2       Oulu   0.221538   57.269850     0.567692  179.795599  2.938462   
3    Tampere   0.369231  103.234448     0.606154  211.767307  2.723077   
4  Mariehamn   0.264615   83.272083     0.535385  181.847994  1.676923   

        CO2_PV  
0  1435.480769  
1  1181.154785  
2  1869.813341  
3  2016.721215  
4  1317.929515  

Final Results:
         City  Best                          Scores
0       Turku  Wind  [1.0, 0.9030033194916672, 0.0]
1    Helsinki  Wind  [1.0, 0.8395564917345069, 0.0]
2        Oulu  Wind  [1.0, 0.9003397302561926, 0.0]
3     Tampere  Wind  [1.0, 0.9211176690689659, 0.0]
4   Mariehamn  Wind   [1.0, 0.863129356466213, 0.0]
5     Aalborg  Wind  [1.0, 0.7695389482679655, 0.0]
6      Aarhus  Wind  [1.0, 0.7020772099698775, 0.0]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>